# RAG + CI Baseline for Clinical Calculators

Retrieval-augmented baseline: the calculator list, the parameter schema and the formula are retrieved from the calculator registry and injected into the prompt of each stage; the model computes the result in text. The notebook is generic: set the **dataset** and the **model** in the Configuration cell.

**Requirements:** a CUDA GPU (two GPUs with the default `MAX_MEMORY`); Python packages listed in `requirements.txt`; the dataset CSV and the calculator registry (`name_to_python.json`) referenced in the Configuration cell.

## Evaluation protocol

Part A is identical in all six notebooks of this release (one MCP notebook and five baseline notebooks). Part B lists what is specific to this pipeline. Both parts describe what the code in this notebook does, so that the numbers in the paper can be checked against it.

### A. Common to all pipelines

**Notation.** N is the number of samples after loading the dataset (rows with missing required fields are dropped). S is the set of samples whose selected calculator equals the ground-truth calculator. An *entity* is one ground-truth input variable, i.e. one key of `Relevant Entities`. S' is the set of samples in S that are scored in the extraction stage, and D the set of samples whose final answer is scored (Part B states both).

**Metrics.**

| Metric | Formula |
|---|---|
| CSA, calculator selection accuracy | CSA = \|S\| / N |
| PEA, parameter extraction accuracy | PEA = (matched entities over S') / (ground-truth entities over S') |
| CCA, calculation correctness accuracy | CCA = \|C\| / \|D\|, C = samples in D whose final answer is correct |

A final answer is correct when the number lies inside the dataset interval `[Lower Limit, Upper Limit]` (inclusive). The category-wise evaluation applies the same three formulas to the samples of one dataset category (`Category` column). CCA uses different denominators in different pipelines (Part B), so it is not directly comparable across pipelines. **No end-to-end metric is computed.**

**Reproducibility settings.**

| Item | Setting |
|---|---|
| Random seed | 42 (Python `random`, NumPy, PyTorch, CUDA) |
| Decoding | greedy; `do_sample=False` in every `generate` call; no temperature, top-p or top-k |
| Quantization | 4-bit NF4 with double quantization, float16 compute dtype (bitsandbytes) |
| Model and dataset | set only in the Configuration cell; nothing else is hard-coded |
| Inputs given to the model | the patient note and the question; ground-truth entities and answers are never part of a prompt |
| Normalization agents | PHONA-S (selection output) and PHONA-E (entity output) are rule-based normalizers, not language models |

**Error taxonomy.** The eight types follow Wang et al. (2025), *From Scores to Steps*. Wang et al. assign them with an LLM judge; this release uses the following deterministic rules, identical in all notebooks (cell *Error taxonomy*). Errors are attributed independently at each stage (not only the first failing stage).

| Type | Counted | Rule |
|---|---|---|
| Formula misselection / hallucination | per sample | the selected calculator differs from the ground truth (N - \|S\| samples, including unmappable outputs) |
| Missing variable | per unmatched entity of S' | no value (empty / null / absent) for the entity |
| Demographic adjustment failure | per unmatched entity | the entity is age, sex / gender, race / ethnicity, pregnancy or BSA |
| Unit conversion error | per unmatched entity | the predicted / ground-truth ratio is within 3 % of a common conversion factor (lb-kg, in-cm, creatinine, glucose, BUN, cholesterol, calcium, bilirubin, factors of 10 and 1000) or the values are related by the Celsius-Fahrenheit formula |
| Clinical misinterpretation | per unmatched entity | the ground-truth value is boolean or categorical (presence, severity, threshold) |
| Incorrect variable extraction | per unmatched entity | any other unmatched numeric value or date |
| Precision error | per sample with a wrong result | the result is outside the dataset interval but within Wang et al.'s rounding tolerance of the ground truth: +-0.05 for answers with at most one decimal place, +-0.005 for two or more |
| Arithmetic error | per sample with a wrong result | any other wrong or non-numeric result |

The rules are tested in this order for extraction errors: missing, demographic, unit conversion, clinical misinterpretation, incorrect extraction. The counts are a diagnostic aid and not metrics; extraction errors are counted per entity, so the total can exceed N.

### B. This pipeline: RAG + CI

Retrieval-augmented prompting at every stage; the retrieval source is the calculator registry (`name_to_python.json` in `CALCS_DIR`) together with the formula and parameter hints defined in the retriever cell. Generation limits: selection 30 new tokens, extraction 100, computation 50; prompts are truncated to 3,000 tokens (selection) and 2,048 tokens (extraction, computation).

**Stage 1, selection (CSA).** The prompt contains the patient note, the question and the retrieved list of all registered calculators with their descriptions. PHONA-S maps the answer to a calculator identifier as in the other baselines (`TOOL_NAME_MAP`, fuzzy matching). A sample is correct when the mapped prediction equals the mapped ground truth. **CSA = |S| / N.**

**Stage 2, extraction (PEA).** Only samples in S are processed. The prompt contains the retrieved parameter schema (type, unit and format hints) of the selected calculator and asks for all fields (an absent boolean is returned as 0). PHONA-E (`normalize_extracted_entities_final`) normalizes the output. An entity is matched when the normalized values are equal or, for numbers, differ by at most 0.01 (at most 0.5 for height and weight). S' = S: a sample with an unusable output contributes all its ground-truth entities as unmatched.

**Stage 3, computation (CCA).** The prompt contains the retrieved formula and instructions, the normalized extracted inputs and the question, and instructs the model to act as a code interpreter: substitute the values, compute step by step and return the final number. The computation is carried out by the model in text; no code is executed in this notebook. The first number in the answer is the prediction. D = S.


## Section 1 — Environment Setup

In [ ]:
import os, json, re, ast, sys, random, warnings
import pandas as pd
import numpy as np
import torch

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("=" * 60)
print("PyTorch:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
print("GPU Count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))
print("=" * 60)

## Section 2 — Dataset and Calculator Paths

In [ ]:
# Configuration: the only cell that needs to be edited.
import os

DATASET_NAME = '<dataset-name>'                 # label used in output paths, e.g. 'medcalc', 'abacus212', 'abacus409'
DATASET_FILE = 'data/<dataset_file>.csv'        # dataset CSV (see README for the expected columns)
MODEL_ID     = '<hf-model-id>'                  # any Hugging Face chat model, e.g. 'org/model-name'
CALCS_DIR    = 'calculators'                    # folder with the calculator .py files and name_to_python.json
MAX_MEMORY   = {0: '14GiB', 1: '14GiB'}         # per-GPU memory cap used when loading the model; None = automatic
RESULTS_DIR  = f'results/rag_ci_{DATASET_NAME}_' + MODEL_ID.split('/')[-1]

os.makedirs(RESULTS_DIR, exist_ok=True)
checks = [(DATASET_FILE, 'Dataset')] + [(CALCS_DIR, 'Calculators')]
for path, label in checks:
    print(('Found   ' if os.path.exists(path) else 'MISSING '), ':', label, '->', path)

## Section 3 — Load and Clean Dataset

In [ ]:
data_raw_df = pd.read_csv(DATASET_FILE)
data_df = data_raw_df[[
    "Calculator Name", "Category", "Patient Note", "Question",
    "Relevant Entities", "Ground Truth Answer", "Lower Limit", "Upper Limit"
]].dropna().reset_index(drop=True)

print("=" * 60)
print("Total Samples:", len(data_df))
print("Unique Calculators:", data_df["Calculator Name"].nunique())
print("Categories:", data_df["Category"].unique().tolist())
print("=" * 60)

## Section 4 — Tool Name Normalizer (TOOL_NAME_MAP)

In [ ]:
from difflib import get_close_matches

TOOL_NAME_MAP = {
    "APACHE II Score": "apache_ii",
    "Adjusted Body Weight": "adjusted_body_weight",
    "Ideal Body Weight": "ideal_body_weight",
    "Albumin Corrected Anion Gap": "albumin_corrected_anion",
    "Albumin Corrected Delta Gap": "albumin_corrected_delta_gap",
    "Albumin Corrected Delta Ratio": "albumin_delta_ratio",
    "Anion Gap": "anion_gap",
    "Delta Gap": "delta_gap",
    "Delta Ratio": "delta_ratio",
    "Body Mass Index (BMI)": "bmi_calculator",
    "Body Surface Area Calculator": "bsa_calculator",
    "CHA2DS2-VASc Score for Atrial Fibrillation Stroke Risk": "cha2ds2_vasc_score",
    "Revised Cardiac Risk Index for Pre-Operative Risk": "cardiac_risk_index",
    "Mean Arterial Pressure (MAP)": "mean_arterial_pressure",
    "Child-Pugh Score for Cirrhosis Mortality": "child_pugh_score",
    "MELD Na (UNOS/OPTN)": "meldna",
    "Glasgow Coma Score (GCS)": "glasgow_coma_score",
    "Sequential Organ Failure Assessment (SOFA) Score": "sofa",
    "Charlson Comorbidity Index (CCI)": "cci",
    "Wells\' Criteria for DVT": "wells_criteria_dvt",
    "Wells\' Criteria for Pulmonary Embolism": "wells_criteria_pe",
    "CKD-EPI Equations for Glomerular Filtration Rate": "ckd-epi_2021_creatinine",
    "Creatinine Clearance (Cockcroft-Gault Equation)": "creatinine_clearance",
    "MDRD GFR Equation": "mdrd_gfr",
    "Fractional Excretion of Sodium (FENa)": "compute_fena",
    "CURB-65 Score for Pneumonia Severity": "curb_65",
    "Caprini Score for Venous Thromboembolism (2005)": "caprini_score",
    "Centor Score (Modified/McIsaac) for Strep Pharyngitis": "centor_score",
    "FeverPAIN Score for Strep Pharyngitis": "feverpain",
    "PSI Score: Pneumonia Severity Index for CAP": "psi_score",
    "SIRS Criteria": "sirs_criteria",
    "Calcium Correction for Hypoalbuminemia": "calcium_correction",
    "LDL Calculated": "ldl_calculated",
    "Serum Osmolality": "sOsm",
    "Sodium Correction for Hyperglycemia": "sodium_correction_hyperglycemia",
    "HOMA-IR (Homeostatic Model Assessment for Insulin Resistance)": "homa_ir",
    "Free Water Deficit": "free_water_deficit",
    "Glasgow-Blatchford Bleeding Score (GBS)": "glasgow_bleeding_score",
    "HAS-BLED Score for Major Bleeding Risk": "has_bled_score",
    "HEART Score for Major Cardiac Events": "heart_score",
    "Framingham Risk Score for Hard Coronary Heart Disease": "framingham_risk_score",
    "PERC Rule for Pulmonary Embolism": "perc_rule",
    "Fibrosis-4 (FIB-4) Index for Liver Fibrosis": "fibrosis_4",
    "Estimated Date of Conception": "estimated_conception_date",
    "Estimated Due Date": "estimated_due_date",
    "Estimated Gestational Age": "estimated_gestational_age",
    "Morphine Milligram Equivalents (MME) Calculator": "mme",
    "Steroid Conversion Calculator": "steroid_conversion_calculator",
    "Maintenance Fluids Calculations": "maintenance_fluid_calc",
    "QTc Bazett Calculator": "qt_calculator_bazett",
    "QTc Framingham Calculator": "qt_calculator_framingham",
    "QTc Fridericia Calculator": "qt_calculator_fredericia",
    "QTc Hodges Calculator": "qt_calculator_hodges",
    "QTc Rautaharju Calculator": "qt_calculator_rautaharju",
    "Target weight": "target_weight",
    "NIH Stroke Scale (NIHSS)": "nihss",
}

SEMANTIC_TOOL_MAP = {
    "cockcroft gault": "creatinine_clearance",
    "cockroft gault": "creatinine_clearance",
    "creatinine clearance": "creatinine_clearance",
    "ckd epi": "ckd-epi_2021_creatinine",
    "mdrd gfr": "mdrd_gfr",
    "bmi": "bmi_calculator",
    "body mass": "bmi_calculator",
    "charlson": "cci",
    "sofa": "sofa",
    "glasgow coma": "glasgow_coma_score",
    "wells dvt": "wells_criteria_dvt",
    "wells pulmonary": "wells_criteria_pe",
    "wells pe": "wells_criteria_pe",
    "curb 65": "curb_65",
    "child pugh": "child_pugh_score",
    "meld na": "meldna",
    "cha2ds2": "cha2ds2_vasc_score",
    "has bled": "has_bled_score",
    "heart score": "heart_score",
    "framingham risk": "framingham_risk_score",
    "caprini": "caprini_score",
    "centor": "centor_score",
    "feverpain": "feverpain",
    "perc": "perc_rule",
    "fib 4": "fibrosis_4",
    "fibrosis 4": "fibrosis_4",
    "psi pneumonia": "psi_score",
    "sirs": "sirs_criteria",
    "anion gap": "anion_gap",
    "serum osmolality": "sOsm",
    "sodium correction": "sodium_correction_hyperglycemia",
    "free water": "free_water_deficit",
    "homa ir": "homa_ir",
    "ldl": "ldl_calculated",
    "morphine equivalent": "mme",
    "steroid conversion": "steroid_conversion_calculator",
    "maintenance fluid": "maintenance_fluid_calc",
    "qtc bazett": "qt_calculator_bazett",
    "qtc framingham": "qt_calculator_framingham",
    "qtc fridericia": "qt_calculator_fredericia",
    "qtc hodges": "qt_calculator_hodges",
    "qtc rautaharju": "qt_calculator_rautaharju",
    "estimated due date": "estimated_due_date",
    "gestational age": "estimated_gestational_age",
    "estimated conception": "estimated_conception_date",
    "ideal body weight": "ideal_body_weight",
    "adjusted body weight": "adjusted_body_weight",
    "body surface area": "bsa_calculator",
    "apache": "apache_ii",
    "revised cardiac risk": "cardiac_risk_index",
    "mean arterial": "mean_arterial_pressure",
    "fena": "compute_fena",
    "fractional excretion": "compute_fena",
    "calcium correction": "calcium_correction",
    "glasgow bleeding": "glasgow_bleeding_score",
    "gbs": "glasgow_bleeding_score",
    "nihss": "nihss",
    "nih stroke": "nihss",
}

NOISE_WORDS = ["score","calculator","tool","criteria","index","equation","assessment","risk","the","a","an"]

def clean_text(text):
    text = str(text).lower().strip()
    text = re.sub(r"[-_\'():\[\]{}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_camel_case(text):
    text = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", str(text))
    text = re.sub(r"(?<=[A-Z])(?=[A-Z][a-z])", " ", text)
    return text

def remove_noise_words(text):
    return " ".join([w for w in text.split() if w not in NOISE_WORDS])

def canonicalize(text):
    text = split_camel_case(text)
    text = clean_text(text)
    text = remove_noise_words(text)
    return re.sub(r"\s+", " ", text).strip()

def normalize_tool_name(raw_text):
    canonical = canonicalize(raw_text)
    if canonical in SEMANTIC_TOOL_MAP:
        return SEMANTIC_TOOL_MAP[canonical]
    for phrase, tool in SEMANTIC_TOOL_MAP.items():
        if phrase in canonical or canonical in phrase:
            return tool
    for dataset_name, tool in TOOL_NAME_MAP.items():
        if canonicalize(dataset_name) == canonical:
            return tool
    candidates = {canonicalize(k): v for k, v in {**SEMANTIC_TOOL_MAP, **TOOL_NAME_MAP}.items()}
    matches = get_close_matches(canonical, candidates.keys(), n=1, cutoff=0.82)
    if matches:
        return candidates[matches[0]]
    matches2 = get_close_matches(canonical, candidates.keys(), n=1, cutoff=0.65)
    if matches2:
        return candidates[matches2[0]]
    return "UNKNOWN_TOOL"

valid_tool_names = list(TOOL_NAME_MAP.values())
print("=" * 60)
print("Tool Normalizer Ready")
print("Total Mappings:", len(TOOL_NAME_MAP))
print("=" * 60)

## Section 5 — Load name_to_python.json (RAG Knowledge Store)

In [ ]:
# Load the structured knowledge store for RAG retrieval.
with open(os.path.join(CALCS_DIR, "name_to_python.json")) as f:
    NAME_TO_PYTHON = json.load(f)

SKIP_KEYS = {"file path", "explanation function", "calculator name", "type", "question"}

# Build CALC_REGISTRY: calc_name → {file, function, params, question}
CALC_REGISTRY = {}
for key, info in NAME_TO_PYTHON.items():
    calc_name = info.get("calculator name", "")
    CALC_REGISTRY[calc_name] = {
        "file":     info.get("file path", ""),
        "function": info.get("explanation function", ""),
        "params":   {k: v for k, v in info.items() if k not in SKIP_KEYS},
        "question": info.get("question", "")
    }

print("=" * 60)
print("RAG Knowledge Store Loaded")
print("Calculators in Registry:", len(CALC_REGISTRY))
print("=" * 60)

## Section 6 — Build RAG Retriever Functions

In [ ]:
# ── RAG Retriever Functions ──────────────────────────────────

MANUAL_FORMULA_MAP = {
    "Body Mass Index (BMI)":
        "BMI = weight (kg) / height (m)^2",
    "Steroid Conversion Calculator":
        "Apply steroid conversion table: Dexamethasone 1mg = Prednisone 6.67mg = Methylprednisolone 5.33mg = Hydrocortisone 26.7mg = Cortisone 33.3mg",
    "Morphine Milligram Equivalents (MME) Calculator":
        "MME = dose (mg/day) × conversion factor. Factors: Codeine=0.15, Hydrocodone=1, Morphine=1, Oxycodone=1.5, Hydromorphone=5, Fentanyl patch(mcg/hr)=2.4",
    "Estimated Gestational Age":
        "Gestational Age = (Current Date - Last Menstrual Period) in weeks and days",
    "NIH Stroke Scale (NIHSS)":
        "NIHSS = Sum of 11 items: Consciousness(0-3), Gaze(0-2), Visual(0-3), Facial palsy(0-3), Motor arm(0-4 each), Motor leg(0-4 each), Ataxia(0-2), Sensory(0-2), Language(0-3), Dysarthria(0-2), Extinction(0-2)",
}

# Type + unit + format hints for common parameters.
PARAM_HINTS = {
    "age":              ("number", "years", "e.g. 65"),
    "sex":              ("string", "",      "\"Male\" or \"Female\""),
    "weight":           ("number", "kg",    "e.g. 70.0"),
    "height":           ("number", "cm",    "e.g. 170.0"),
    "creatinine":       ("number", "mg/dL", "e.g. 1.2"),
    "albumin":          ("number", "g/dL",  "e.g. 4.0"),
    "sodium":           ("number", "mEq/L", "e.g. 140"),
    "potassium":        ("number", "mEq/L", "e.g. 4.0"),
    "chloride":         ("number", "mEq/L", "e.g. 100"),
    "bicarbonate":      ("number", "mEq/L", "e.g. 24"),
    "bilirubin":        ("number", "mg/dL", "e.g. 1.0"),
    "blood_urea_nitrogen": ("number", "mg/dL", "e.g. 18"),
    "glucose":          ("number", "mg/dL", "e.g. 100"),
    "calcium":          ("number", "mg/dL", "e.g. 9.5"),
    "platelets":        ("number", "10^9/L","e.g. 200"),
    "heart_rate":       ("number", "bpm",   "e.g. 80"),
    "systolic_bp":      ("number", "mmHg",  "e.g. 120"),
    "diastolic_bp":     ("number", "mmHg",  "e.g. 80"),
    "temperature":      ("number", "°C",    "e.g. 37.0"),
    "respiratory_rate": ("number", "breaths/min", "e.g. 16"),
    "oxygen_saturation":("number", "%",     "e.g. 98"),
    "inr":              ("number", "",      "e.g. 1.1"),
    "ptt":              ("number", "seconds","e.g. 30"),
    "white_blood_cell_count": ("number", "10^9/L", "e.g. 8.0"),
    "menstrual_date":   ("string", "",      "MM/DD/YYYY format"),
    "conception_date":  ("string", "",      "MM/DD/YYYY format"),
    "current_date":     ("string", "",      "MM/DD/YYYY format"),
}

# Boolean params — common in risk/severity calculators.
BOOLEAN_PARAM_KEYWORDS = [
    "chf","diabetes","hypertension","stroke","vascular","female",
    "age65","age75","smoking","dyslipidemia","family_history",
    "chest_pain","diaphoresis","history","prior","known","active",
    "mechanical","atrial","cancer","paralysis","copd","renal",
    "liver","ischemia","abnormal","altered","confusion","fever",
    "tachycardia","leukocytosis","bandemia","immobility","surgery",
    "trauma","pregnancy","obesity","cardiac","pulmonary","sepsis",
]


def retrieve_calculator_list():
    """Step 1 RAG: Retrieve all calculator names + descriptions."""
    lines = []
    for calc_name, info in CALC_REGISTRY.items():
        description = info.get("question", f"Calculate {calc_name}")
        description = description.split(".")[0][:80]
        lines.append(f"- {calc_name}: {description}")
    return "\n".join(lines)


def retrieve_input_parameters(calc_name):
    """
    Step 2 RAG:
    Retrieve required input parameters WITH type, unit, and format hints.
    Helps SLM extract in correct format — reduces PEA errors.
    """
    info   = CALC_REGISTRY.get(calc_name, {})
    params = info.get("params", {})
    if not params:
        return "No specific parameters found."

    lines = []
    for display_key, param_key in params.items():
        # Check if boolean parameter.
        is_bool = any(kw in param_key.lower() for kw in BOOLEAN_PARAM_KEYWORDS)

        if is_bool:
            line = f"- {display_key} ({param_key}): boolean — 1 if present/yes, 0 if absent/no"
        elif param_key in PARAM_HINTS:
            ptype, punit, pexample = PARAM_HINTS[param_key]
            unit_str = f", unit: {punit}" if punit else ""
            line = f"- {display_key} ({param_key}): {ptype}{unit_str} — {pexample}"
        elif "date" in param_key.lower():
            line = f"- {display_key} ({param_key}): string — MM/DD/YYYY format"
        else:
            line = f"- {display_key} ({param_key}): number"

        lines.append(line)

    return "\n".join(lines)


def extract_formula_from_py(py_path):
    """Extract formula string from .py calculator file."""
    try:
        with open(py_path, "r") as f:
            content = f.read()
        for line in content.split("\n"):
            s = line.strip()
            if re.search(r"formula|equation|given by|computed by|calculated", s, re.IGNORECASE):
                m = re.search(r'["\'\']((?:[^"\'\'\\]|\\.){15,300})["\'\']', s)
                if m and len(m.group(1)) > 20:
                    return m.group(1)[:250]
        for line in content.split("\n"):
            s = line.strip()
            if "plugging" in s.lower() or "result" in s.lower():
                m = re.search(r'["\'\']((?:[^"\'\'\\]|\\.){15,300})["\'\']', s)
                if m and "=" in m.group(1):
                    return m.group(1)[:250]
        criteria_lines = []
        for line in content.split("\n"):
            s = line.strip()
            if re.match(r"\d+\.", s) and "point" in s.lower():
                criteria_lines.append(s)
        if criteria_lines:
            return " | ".join(criteria_lines[:4])[:250]
        return None
    except:
        return None


def retrieve_formula(calc_name):
    """
    Step 3 RAG:
    Retrieve formula WITH step-by-step instructions.
    Reduces CCA arithmetic errors.
    """
    info      = CALC_REGISTRY.get(calc_name, {})
    file_name = info.get("file", "")
    question  = info.get("question", "")

    if calc_name in MANUAL_FORMULA_MAP:
        formula      = MANUAL_FORMULA_MAP[calc_name]
        instructions = question[:200] if question else f"Compute the {calc_name}."
        return formula, instructions

    if file_name:
        py_path   = os.path.join(CALCS_DIR, file_name)
        extracted = extract_formula_from_py(py_path)
        if extracted:
            instructions = question[:200] if question else f"Compute the {calc_name}."
            return extracted, instructions

    if question:
        return f"Apply the {calc_name} formula as described.", question[:200]

    return f"Apply the standard {calc_name} clinical formula.", f"Compute the {calc_name}."


RETRIEVED_CALC_LIST = retrieve_calculator_list()

print("=" * 60)
print("RAG Retriever Functions Built")
print("Param hints defined:", len(PARAM_HINTS))
print("Boolean keywords:", len(BOOLEAN_PARAM_KEYWORDS))
print("Manual formulas:", len(MANUAL_FORMULA_MAP))
print()
print("Sample extraction CI (Creatinine Clearance):")
print(retrieve_input_parameters("Creatinine Clearance (Cockcroft-Gault Equation)"))
print("=" * 60)

## Section 7 — Load Model (4-bit)

In [ ]:
# Load the model with 4-bit NF4 quantization (dependencies: see requirements.txt).
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from huggingface_hub import login

if os.environ.get('HF_TOKEN'):   # only needed for gated models; never store a token in the notebook
    login(token=os.environ['HF_TOKEN'])

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

slm_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
slm_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    max_memory=MAX_MEMORY,
    trust_remote_code=True,
)
slm_model.eval()
print(MODEL_ID, "loaded.")

## Section 8 — Step 1: Calculator Selection with RAG

In [ ]:
# ── RAG Selection Prompt (User-defined) ─────────────────────

SELECTION_SYSTEM = (
    "You are a clinical calculator selection assistant. "
    "You will be provided a list of relevant calculators "
    "with descriptions. Use only this list to select "
    "the most appropriate calculator."
)

def run_rag_selection(patient_note, question):
    """
    Step 1: RAG-augmented calculator selection.
    CI = retrieved calculator list injected into prompt.
    """
    user_prompt = (
        f"RELEVANT CALCULATORS:\n"
        f"{RETRIEVED_CALC_LIST}\n\n"
        f"Patient Note: {patient_note}\n"
        f"Question: {question}\n"
        f"From the list above, return only the most "
        f"appropriate calculator name."
    )
    messages = [
        {"role": "system", "content": SELECTION_SYSTEM},
        {"role": "user",   "content": user_prompt}
    ]
    prompt_text = slm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = slm_tokenizer(
        prompt_text, return_tensors="pt",
        max_length=3000, truncation=True
    )
    inputs = {k: v.to(slm_model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = slm_model.generate(
            **inputs, max_new_tokens=30, do_sample=False,
            pad_token_id=slm_tokenizer.eos_token_id
        )
    raw = slm_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return raw, normalize_tool_name(raw)

print("RAG Selection function ready.")

In [ ]:
# ── Run Step 1: Selection on all samples ──────────────

raw_sel_outputs, norm_sel_outputs = [], []
gt_tools, sel_correct = [], []
TOTAL = len(data_df)

print("=" * 60)
print("Step 1: RAG Calculator Selection Started")
print("Total Samples:", TOTAL)
print("=" * 60)

for idx in range(TOTAL):
    row = data_df.iloc[idx]
    patient_note = row["Patient Note"]
    question     = row["Question"]
    gt_calc      = row["Calculator Name"]

    if idx % 10 == 0:
        torch.cuda.empty_cache()

    raw_out, norm_pred = run_rag_selection(patient_note, question)

    # Normalize GT tool name using TOOL_NAME_MAP.
    gt_tool    = TOOL_NAME_MAP.get(gt_calc, "UNKNOWN_TOOL")
    is_correct = (norm_pred == gt_tool)

    raw_sel_outputs.append(raw_out)
    norm_sel_outputs.append(norm_pred)
    gt_tools.append(gt_tool)
    sel_correct.append(is_correct)

    if (idx + 1) % 25 == 0:
        print(f"Processed: {idx+1}/{TOTAL}")

# ── CSA Evaluation ────────────────────────────────────────────
S_set       = [i for i, c in enumerate(sel_correct) if c]
CSA_accuracy = len(S_set) / TOTAL * 100
unknown_count = norm_sel_outputs.count("UNKNOWN_TOOL")

print("=" * 60)
print(f"CSA Selection Accuracy: {round(CSA_accuracy, 2)} %")
print(f"Correct |S|: {len(S_set)} / {TOTAL}")
print(f"UNKNOWN_TOOL: {unknown_count}")
print("=" * 60)

# Save selection results.
sel_df = pd.DataFrame({
    "sample_id":         range(TOTAL),
    "category":          data_df["Category"].tolist(),
    "calculator_name":   data_df["Calculator Name"].tolist(),
    "gt_tool":           gt_tools,
    "raw_output":        raw_sel_outputs,
    "predicted_tool":    norm_sel_outputs,
    "is_correct":        sel_correct
})
sel_df.to_csv(os.path.join(RESULTS_DIR, "selection_results.csv"), index=False)
print("Selection results saved.")

## Section 9 — Build Entity Normalizer (PHONA-E)

In [ ]:
# ── PHONA-E: Full Entity Normalizer (from reference notebook) ──

import re, json, ast

MEDICAL_ABBREVIATIONS = {
    "cr":"creatinine","scr":"creatinine","bun":"blood_urea_nitrogen",
    "na":"sodium","k":"potassium","cl":"chloride",
    "hco3":"bicarbonate","co2":"bicarbonate",
    "hr":"heart_rate","rr":"respiratory_rate",
    "bp":"blood_pressure","sbp":"systolic_bp","dbp":"diastolic_bp",
    "spo2":"oxygen_saturation","hb":"hemoglobin","hgb":"hemoglobin",
    "wbc":"white_blood_cell_count","plt":"platelets",
    "gcs":"glasgow_coma_score","inr":"inr"
}

ENTITY_ALIAS_MAP = {
    "patient_age":"age","age_years":"age",
    "gender":"sex","patient_gender":"sex","patient_sex":"sex",
    "patient_weight":"weight","body_weight":"weight","weight_kg":"weight",
    "patient_height":"height","height_cm":"height",
    "serum_creatinine":"creatinine","creatinine_mg_dl":"creatinine",
    "serum_sodium":"sodium","serum_na":"sodium",
    "serum_potassium":"potassium","serum_k":"potassium",
    "serum_chloride":"chloride","serum_cl":"chloride",
    "serum_bicarbonate":"bicarbonate","bicarbonate_level":"bicarbonate",
    "blood_urea_nitrogen_bun":"blood_urea_nitrogen",
    "serum_glucose":"glucose","blood_glucose":"glucose",
    "fasting_glucose":"glucose","serum_insulin":"insulin",
    "fasting_insulin":"insulin","serum_albumin":"albumin",
    "serum_calcium":"calcium","patient_calcium":"calcium",
    "total_bilirubin":"bilirubin","serum_bilirubin":"bilirubin",
    "heart_rate_or_pulse":"heart_rate","pulse":"heart_rate",
    "patient_heart_rate":"heart_rate","temp":"temperature",
    "respiratory_rate":"respiratory_rate",
    "oxygen_saturation":"oxygen_saturation","o2_saturation":"oxygen_saturation",
    "platelet_count":"platelets",
    "white_blood_cells":"white_blood_cell_count",
    "glasgow_coma_scale":"glasgow_coma_score",
}

NOISE_FIELDS = {
    "calculator","tool","tool_name","name","version",
    "equation","description","source","points",
    "score","output","metadata","status"
}

def remove_markdown_json(text):
    text = str(text).strip()
    text = text.replace("```json","").replace("```JSON","").replace("```","")
    return text.strip()

def extract_json_block(text):
    text = remove_markdown_json(text)
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else text

def repair_json_text(text):
    text = extract_json_block(text)
    text = re.sub(r"\bNone\b","null",text)
    text = re.sub(r"\bTrue\b","true",text)
    text = re.sub(r"\bFalse\b","false",text)
    text = re.sub(r",\s*}","}",text)
    text = re.sub(r",\s*]","]",text)
    return text

def parse_extraction_output(text):
    repaired = repair_json_text(text)
    try: return json.loads(repaired)
    except:
        try: return ast.literal_eval(repaired)
        except: return {}

def clean_entity_key(key):
    key = str(key).strip().lower()
    key = re.sub(r"[^a-z0-9]+","_",key)
    key = re.sub(r"_+","_",key).strip("_")
    return key

def normalize_entity_key_final(key):
    key = clean_entity_key(key)
    key = MEDICAL_ABBREVIATIONS.get(key, key)
    key = ENTITY_ALIAS_MAP.get(key, key)
    return key

def normalize_boolean_value(value):
    v = str(value).strip().lower()
    if v in {"true","yes","present","positive","1","y"}: return True
    if v in {"false","no","absent","negative","0","n"}:  return False
    return value

def normalize_categorical_value(value):
    v = str(value).strip().lower()
    if v in {"m","male","man"}:    return "male"
    if v in {"f","female","woman"}: return "female"
    return value

def extract_numeric_value(value):
    if isinstance(value, list) and len(value) > 0:
        value = value[0]
    match = re.search(r"-?\d+\.?\d*", str(value).strip().lower())
    if match:
        try: return float(match.group(0))
        except: return value
    return value

def normalize_unit_value(value, entity_key=None):
    unit = ""
    if isinstance(value, list):
        numeric_value = value[0] if len(value) > 0 else value
        unit = str(value[1]).lower() if len(value) > 1 else ""
    else:
        numeric_value = value
    numeric_value = extract_numeric_value(numeric_value)
    if isinstance(numeric_value, float):
        if entity_key == "height":
            if "m" in unit and "cm" not in unit:
                return round(numeric_value * 100, 4)
            if numeric_value < 3:
                return round(numeric_value * 100, 4)
        return round(numeric_value, 4)
    return numeric_value

def normalize_entity_value_final(value, entity_key=None):
    value = normalize_boolean_value(value)
    value = normalize_categorical_value(value)
    value = normalize_unit_value(value, entity_key=entity_key)
    if isinstance(value, str):
        return value.strip().lower()
    return value

def collect_structured_input_list(data):
    collected = {}
    if isinstance(data, list):
        for item in data:
            if not isinstance(item, dict): continue
            if "name" in item and "value" in item:
                collected[item["name"]] = item["value"]
            elif "parameter" in item and "value" in item:
                collected[item["parameter"]] = item["value"]
    return collected

def recursive_flatten_entities(data):
    collected = {}
    if isinstance(data, dict):
        if "name" in data and "value" in data:
            collected[data["name"]] = data["value"]
            return collected
        if "parameter" in data and "value" in data:
            collected[data["parameter"]] = data["value"]
            return collected
        for key, value in data.items():
            clean_key = clean_entity_key(key)
            if clean_key in NOISE_FIELDS and not isinstance(value,(dict,list)): continue
            if isinstance(value, list):
                structured = collect_structured_input_list(value)
                if structured: collected.update(structured)
                else: collected[key] = value
            elif isinstance(value, dict):
                collected.update(recursive_flatten_entities(value))
            else:
                collected[key] = value
    elif isinstance(data, list):
        collected.update(collect_structured_input_list(data))
        for item in data:
            if isinstance(item, dict):
                collected.update(recursive_flatten_entities(item))
    return collected

def remove_noise_fields_dict(entity_dict):
    return {k: v for k, v in entity_dict.items()
            if clean_entity_key(k) not in NOISE_FIELDS}

def values_match_final(true_value, predicted_value, entity_key=None):
    """Full value normalization — handles boolean, numeric, categorical."""
    true_norm = normalize_entity_value_final(true_value,      entity_key=entity_key)
    pred_norm = normalize_entity_value_final(predicted_value, entity_key=entity_key)
    if true_norm == pred_norm:
        return True
    if isinstance(true_norm, float) and isinstance(pred_norm, float):
        diff = abs(true_norm - pred_norm)
        if diff <= 0.01: return True
        if entity_key in ["height","weight"] and diff <= 0.5: return True
    return False

def normalize_extracted_entities_final(raw_output):
    """Full PHONA-E pipeline — same as reference notebook."""
    parsed = parse_extraction_output(raw_output)
    if not isinstance(parsed, (dict,list)): return {}
    flat = recursive_flatten_entities(parsed)
    flat = remove_noise_fields_dict(flat)
    result = {}
    for key, value in flat.items():
        norm_key = normalize_entity_key_final(key)
        norm_val = normalize_entity_value_final(value, entity_key=norm_key)
        result[norm_key] = norm_val
    return result

print("=" * 60)
print("PHONA-E built successfully (full reference version).")
print("values_match_final: handles boolean + numeric + categorical.")
print("=" * 60)

## Section 10 — Step 2: Entity Extraction with RAG

In [ ]:
# ── RAG Extraction Prompt ────────────────────────

EXTRACTION_SYSTEM = (
    "You are a clinical data extraction assistant. "
    "You will be provided the required input parameters with types and formats. "
    "Extract ALL listed parameters from the patient note. "
    "If a boolean parameter is absent or not mentioned, write 0. "
    "If a numeric parameter is not mentioned, write null. "
    "Return a valid flat JSON object. Nothing else."
)

def run_rag_extraction(calc_name, patient_note):
    retrieved_inputs = retrieve_input_parameters(calc_name)
    user_prompt = (
        f"CALCULATOR INFORMATION:\n"
        f"Name: {calc_name}\n\n"
        f"Required Inputs (extract ALL — use 0 for absent booleans):\n"
        f"{retrieved_inputs}\n\n"
        f"Patient Note: {patient_note}\n\n"
        f"Rules:\n"
        f"- Extract EVERY parameter listed above.\n"
        f"- Boolean: 1 if present/yes, 0 if absent/not mentioned.\n"
        f"- Number: exact value from note, null if not found.\n"
        f"- Date: MM/DD/YYYY format.\n"
        f"- Sex: \"Male\" or \"Female\".\n"
        f"Return flat JSON only."
    )
    messages = [
        {"role": "system", "content": EXTRACTION_SYSTEM},
        {"role": "user",   "content": user_prompt}
    ]
    prompt_text = slm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = slm_tokenizer(
        prompt_text, return_tensors="pt",
        max_length=2048, truncation=True
    )
    inputs = {k: v.to(slm_model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = slm_model.generate(
            **inputs, max_new_tokens=100, do_sample=False,
            pad_token_id=slm_tokenizer.eos_token_id
        )
    raw = slm_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return raw

print("RAG Extraction function ready.")
print("All fields are forced; absent booleans = 0.")

In [ ]:
# ── Run Step 2: Extraction on i ∈ S only ────────────────────

raw_ext_outputs, norm_ext_outputs = [], []
ext_sample_ids, ext_categories, ext_calc_names = [], [], []
ext_matched, ext_total = [], []

print("=" * 60)
print("Step 2: RAG Entity Extraction Started")
print("Samples (i in S):", len(S_set))
print("=" * 60)

for count, idx in enumerate(S_set):
    row          = data_df.iloc[idx]
    calc_name    = row["Calculator Name"]
    patient_note = row["Patient Note"]
    gt_entities_raw = row["Relevant Entities"]

    if count % 10 == 0:
        torch.cuda.empty_cache()

    raw_output = run_rag_extraction(calc_name, patient_note)
    norm_json  = normalize_extracted_entities_final(raw_output)

    # Parse GT entities.
    try:
        gt_entities = json.loads(gt_entities_raw)
    except:
        try:
            gt_entities = ast.literal_eval(str(gt_entities_raw))
        except:
            gt_entities = {}

    # Compute matched fields.
    matched = sum(
        1 for k, v in gt_entities.items()
        if values_match_final(v, norm_json.get(normalize_entity_key_final(k)),
                              entity_key=normalize_entity_key_final(k))
    )
    total_fields = len(gt_entities)

    raw_ext_outputs.append(raw_output)
    norm_ext_outputs.append(json.dumps(norm_json))
    ext_sample_ids.append(idx)
    ext_categories.append(row["Category"])
    ext_calc_names.append(calc_name)
    ext_matched.append(matched)
    ext_total.append(total_fields)

    if (count + 1) % 25 == 0:
        print(f"Extracted: {count+1}/{len(S_set)}")

# ── PEA Evaluation ────────────────────────────────────────────
total_matched    = sum(ext_matched)
total_fields_all = sum(ext_total)
PEA_accuracy = (total_matched / total_fields_all * 100) if total_fields_all > 0 else 0

print("=" * 60)
print(f"PEA Extraction Accuracy: {round(PEA_accuracy, 2)} %")
print(f"Matched: {total_matched} / {total_fields_all}")
print("=" * 60)

ext_df = pd.DataFrame({
    "sample_id": ext_sample_ids, "category": ext_categories,
    "calculator_name": ext_calc_names, "raw_output": raw_ext_outputs,
    "normalized_json": norm_ext_outputs,
    "matched_fields": ext_matched, "total_fields": ext_total
})
ext_df.to_csv(os.path.join(RESULTS_DIR, "extraction_results.csv"), index=False)
print("Extraction results saved.")

## Section 11 — Step 3: Computation with RAG + Code Interpreter (CI)

In [ ]:
# ── RAG + CI Computation (RAG only — no Tool Executor) ────────

COMPUTATION_SYSTEM = (
    "You are a clinical calculator assistant. "
    "You will be provided the exact formula and all required inputs. "
    "Step 1: Substitute the input values into the formula. "
    "Step 2: Compute the result step by step. "
    "Step 3: Return ONLY the final numerical answer (number only)."
)

def extract_numeric_answer(text):
    if text is None: return None
    try: return float(str(text).strip())
    except: pass
    match = re.search(r"-?\d+\.?\d*", str(text))
    return float(match.group(0)) if match else None

def run_rag_ci_computation(calc_name, extracted_json_str, question):
    """
    RAG + CI Computation:
    - RAG: retrieve formula + inject into prompt.
    - CI: SLM computes step by step using provided formula.
    """
    retrieved_formula, retrieved_instructions = retrieve_formula(calc_name)

    try:
        extracted_dict = json.loads(extracted_json_str)
        inputs_str = ", ".join([f"{k}={v}" for k,v in extracted_dict.items()])
    except:
        inputs_str = extracted_json_str

    user_prompt = (
        f"CALCULATOR: {calc_name}\n\n"
        f"FORMULA:\n{retrieved_formula}\n\n"
        f"INSTRUCTIONS:\n{retrieved_instructions}\n\n"
        f"EXTRACTED INPUTS:\n{inputs_str}\n\n"
        f"QUESTION: {question}\n\n"
        f"Step 1: Substitute values into formula.\n"
        f"Step 2: Compute each part of the formula carefully.\n"
        f"Step 3: Apply any conditional rules (e.g. if female multiply by 0.85).\n"
        f"Step 4: Return ONLY the final numerical answer."
    )
    messages = [
        {"role": "system", "content": COMPUTATION_SYSTEM},
        {"role": "user",   "content": user_prompt}
    ]
    prompt_text = slm_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = slm_tokenizer(
        prompt_text, return_tensors="pt",
        max_length=2048, truncation=True
    )
    inputs = {k: v.to(slm_model.device) for k, v in inputs.items()}
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = slm_model.generate(
            **inputs, max_new_tokens=50, do_sample=False,
            pad_token_id=slm_tokenizer.eos_token_id
        )
    raw = slm_tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return raw

print("=" * 60)
print("RAG + CI Computation ready.")
print("RAG: formula retrieved and injected.")
print("CI:  SLM computes step by step.")
print("=" * 60)

In [ ]:
# ── Run Step 3: Computation on i ∈ S only ───────────────────

comp_sample_ids, comp_categories, comp_calc_names = [], [], []
comp_raw_outputs, comp_predicted = [], []
comp_gt, comp_lower, comp_upper, comp_correct = [], [], [], []

print("=" * 60)
print("Step 3: RAG + CI Computation Started")
print("Samples (i in S):", len(S_set))
print("=" * 60)

for count, idx in enumerate(S_set):
    row          = data_df.iloc[idx]
    calc_name    = row["Calculator Name"]
    question     = row["Question"]
    gt_ans       = row["Ground Truth Answer"]
    lower        = row["Lower Limit"]
    upper        = row["Upper Limit"]

    if count % 10 == 0:
        torch.cuda.empty_cache()

    # Get normalized JSON from extraction step.
    ext_idx = ext_sample_ids.index(idx) if idx in ext_sample_ids else -1
    if ext_idx == -1:
        comp_correct.append(False)
        comp_sample_ids.append(idx); comp_categories.append(row["Category"])
        comp_calc_names.append(calc_name); comp_raw_outputs.append("")
        comp_predicted.append(None); comp_gt.append(gt_ans)
        comp_lower.append(lower); comp_upper.append(upper)
        continue

    extracted_json_str = norm_ext_outputs[ext_idx]

    # Run RAG + CI computation.
    raw_output = run_rag_ci_computation(calc_name, extracted_json_str, question)
    predicted  = extract_numeric_answer(raw_output)

    # CCA: Check if within range.
    lower_n = extract_numeric_answer(str(lower))
    upper_n = extract_numeric_answer(str(upper))
    is_correct = False
    if predicted is not None and lower_n is not None and upper_n is not None:
        is_correct = (lower_n <= predicted <= upper_n)

    comp_sample_ids.append(idx); comp_categories.append(row["Category"])
    comp_calc_names.append(calc_name); comp_raw_outputs.append(raw_output)
    comp_predicted.append(predicted); comp_gt.append(gt_ans)
    comp_lower.append(lower); comp_upper.append(upper)
    comp_correct.append(is_correct)

    if (count + 1) % 25 == 0:
        print(f"Computed: {count+1}/{len(S_set)}")

# ── CCA Evaluation ────────────────────────────────────────────
correct_count = sum(comp_correct)
CCA_accuracy   = (correct_count / len(S_set) * 100) if S_set else 0


print("=" * 60)
print(f"CCA Computation Accuracy: {round(CCA_accuracy, 2)} %  |C|={correct_count}/|S|={len(S_set)}")
print("=" * 60)

comp_df = pd.DataFrame({
    "sample_id": comp_sample_ids, "category": comp_categories,
    "calculator_name": comp_calc_names, "raw_output": comp_raw_outputs,
    "predicted_answer": comp_predicted, "ground_truth_answer": comp_gt,
    "lower_limit": comp_lower, "upper_limit": comp_upper, "is_correct": comp_correct
})
comp_df.to_csv(os.path.join(RESULTS_DIR, "computation_results.csv"), index=False)
print("Computation results saved.")

## Error analysis, category-wise evaluation and summary

The next three cells are identical in all notebooks except for the first one, which collects the per-sample results of this pipeline.

In [ ]:
# Per-sample evaluation table (inputs of the error analysis and of the category-wise evaluation).
import ast, json
import pandas as pd

sel = list(sel_correct)
ext_info, ext_unmatched = {}, []
for ei, idx in enumerate(ext_sample_ids):
    raw_gt = data_df.iloc[idx]["Relevant Entities"]
    try:
        gt_ent = json.loads(raw_gt)
    except Exception:
        try:
            gt_ent = ast.literal_eval(str(raw_gt))
        except Exception:
            gt_ent = {}
    pred_ent = json.loads(norm_ext_outputs[ei])
    matched = 0
    for gt_key, gt_val in gt_ent.items():
        norm_key = normalize_entity_key_final(gt_key)
        if values_match_final(gt_val, pred_ent.get(norm_key), entity_key=norm_key):
            matched += 1
        else:
            ext_unmatched.append({"sample_id": idx, "category": data_df.iloc[idx]["Category"], "key": norm_key,
                                  "gt": gt_val, "pred": pred_ent.get(norm_key)})
    ext_info[idx] = (len(gt_ent), matched)

comp_info, comp_wrong = {}, []
for ci, idx in enumerate(comp_sample_ids):
    ok = bool(comp_correct[ci])
    comp_info[idx] = ok
    if not ok:
        comp_wrong.append({"sample_id": idx, "category": data_df.iloc[idx]["Category"],
                           "pred": comp_predicted[ci], "gt": comp_gt[ci]})
eval_df = pd.DataFrame([{
    "sample_id": i, "category": data_df.iloc[i]["Category"], "calculator": data_df.iloc[i]["Calculator Name"],
    "selected_correct": bool(sel[i]),
    "ext_evaluated": i in ext_info, "n_entities": ext_info.get(i, (0, 0))[0], "n_matched": ext_info.get(i, (0, 0))[1],
    "comp_evaluated": i in comp_info, "comp_correct": bool(comp_info.get(i, False)),
} for i in range(len(data_df))])

In [ ]:
# Error taxonomy: rule-based operationalization of the categories of Wang et al. (2025), "From Scores to Steps".
# The same rules are used in all six notebooks.
import re

ERROR_TYPES = [
    "formula_misselection_hallucination",  # selected calculator differs from the ground-truth calculator
    "incorrect_variable_extraction",       # wrong value of a numeric variable (or date)
    "missing_variable",                    # required variable absent from the extraction
    "clinical_misinterpretation",          # wrong boolean / categorical value (presence, severity, threshold)
    "demographic_adjustment_failure",      # wrong or missing age, sex, race, BSA or pregnancy variable
    "arithmetic_error",                    # formula and inputs right, result wrong beyond rounding
    "unit_conversion_error",               # value differs from the ground truth by a unit-conversion factor
    "precision_error",                     # result within the rounding tolerance of the ground truth but outside the dataset interval
]
DEMOGRAPHIC_TOKENS = {"age", "sex", "gender", "race", "ethnicity", "pregnancy", "pregnant", "bsa"}
UNIT_FACTORS = [2.20462, 0.45359, 2.54, 0.3937, 88.4, 0.01131, 18.0182, 0.0555, 2.8, 0.357,
                38.67, 0.02586, 4.008, 0.2495, 17.1, 0.0585, 10.0, 0.1, 1000.0, 0.001]
UNIT_FACTOR_TOLERANCE = 0.03    # relative tolerance when comparing the value ratio with a conversion factor
ROUNDING_TOLERANCE = {1: 0.05, 2: 0.005}   # Wang et al.: +-0.05 for one decimal place, +-0.005 for two
DATE_PATTERN = re.compile(r"^\s*\d{1,2}/\d{1,2}/\d{2,4}")

def scalar(v):
    return v[0] if isinstance(v, (list, tuple)) and len(v) > 0 else v

def to_float(v):
    v = scalar(v)
    if v is None or isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        return float(v)
    s = str(v).strip().replace(",", "")
    if DATE_PATTERN.match(s):
        return None
    m = re.search(r"-?\d+\.?\d*", s)
    return float(m.group(0)) if m else None

def is_missing(v):
    return v is None or str(v).strip().lower() in ("", "none", "null", "nan")

def is_demographic_key(key):
    tokens = set(re.split(r"[^a-z0-9]+", str(key).lower()))
    return bool(tokens & DEMOGRAPHIC_TOKENS) or "body_surface" in str(key).lower() or "surface area" in str(key).lower()

def is_unit_error(gt, pred):
    g, p = to_float(gt), to_float(pred)
    if g is None or p is None or g == 0 or p == 0:
        return False
    ratio = p / g
    if any(abs(ratio / f - 1) <= UNIT_FACTOR_TOLERANCE for f in UNIT_FACTORS):
        return True
    return abs(p - (g * 9 / 5 + 32)) <= 1 or abs(p - (g - 32) * 5 / 9) <= 1   # Celsius <-> Fahrenheit

def classify_extraction_error(key, gt, pred):
    if is_missing(scalar(pred)):
        return "missing_variable"
    if is_demographic_key(key):
        return "demographic_adjustment_failure"
    if is_unit_error(gt, pred):
        return "unit_conversion_error"
    g = scalar(gt)
    if to_float(g) is None and not DATE_PATTERN.match(str(g)):
        return "clinical_misinterpretation"
    return "incorrect_variable_extraction"

def decimals(text):
    s = str(scalar(text)).strip()
    m = re.search(r"\.(\d+)", s)
    return min(len(m.group(1)), 2) if m else 0

def classify_computation_error(pred, gt):
    p, g = to_float(pred), to_float(gt)
    if p is None or g is None:
        return "arithmetic_error"
    tol = ROUNDING_TOLERANCE.get(decimals(pred), ROUNDING_TOLERANCE[1] if decimals(pred) == 0 else 0.005)
    return "precision_error" if abs(p - g) <= tol else "arithmetic_error"

In [ ]:
# Error analysis, category-wise evaluation and overall summary (identical in all notebooks).
# Inputs built by the previous cell: eval_df (one row per sample), ext_unmatched, comp_wrong.
import json, os
import pandas as pd

N = len(eval_df)
n_selected = int(eval_df["selected_correct"].sum())
ext_part = eval_df[eval_df["ext_evaluated"]]
comp_part = eval_df[eval_df["comp_evaluated"]]

def pct(a, b):
    return round(a / b * 100, 2) if b else float("nan")

# ---- overall metrics
CSA = pct(n_selected, N)                                   # |S| / N
PEA = pct(int(ext_part["n_matched"].sum()), int(ext_part["n_entities"].sum()))   # matched / ground-truth entities
CCA = pct(int(comp_part["comp_correct"].sum()), len(comp_part))                  # |C| / |D|

print("=" * 70)
print(f"CSA = |S| / N                : {n_selected} / {N} = {CSA} %")
print(f"PEA = matched / entities     : {int(ext_part['n_matched'].sum())} / {int(ext_part['n_entities'].sum())} = {PEA} %")
print(f"CCA = |C| / |D|              : {int(comp_part['comp_correct'].sum())} / {len(comp_part)} = {CCA} %")
print("=" * 70)

# ---- error analysis
error_counts = {t: 0 for t in ERROR_TYPES}
error_records = []
for _, r in eval_df[~eval_df["selected_correct"]].iterrows():
    error_counts["formula_misselection_hallucination"] += 1
    error_records.append({"sample_id": r["sample_id"], "category": r["category"], "stage": "selection",
                          "error_type": "formula_misselection_hallucination"})
for e in ext_unmatched:
    t = classify_extraction_error(e["key"], e["gt"], e["pred"])
    error_counts[t] += 1
    error_records.append({"sample_id": e["sample_id"], "category": e["category"], "stage": "extraction",
                          "field": e["key"], "error_type": t})
for w in comp_wrong:
    t = classify_computation_error(w["pred"], w["gt"])
    error_counts[t] += 1
    error_records.append({"sample_id": w["sample_id"], "category": w["category"], "stage": "computation",
                          "error_type": t})

total_errors = sum(error_counts.values())
print(f'{"Error type":<40} {"Count":>7} {"Share":>8}')
for t in ERROR_TYPES:
    print(f"{t:<40} {error_counts[t]:>7} {pct(error_counts[t], total_errors):>7}%")
print(f'{"TOTAL":<40} {total_errors:>7}')
print("(formula errors and computation errors are counted per sample, extraction errors per unmatched entity)")

# ---- category-wise evaluation (same formulas as above, restricted to one category)
def stage_table(group_col):
    rows = []
    for g in sorted(eval_df[group_col].dropna().unique(), key=str):
        d = eval_df[eval_df[group_col] == g]
        e = d[d["ext_evaluated"]]
        c = d[d["comp_evaluated"]]
        rows.append({group_col: g, "N": len(d), "S": int(d["selected_correct"].sum()),
                     "CSA_%": pct(int(d["selected_correct"].sum()), len(d)),
                     "PEA_%": pct(int(e["n_matched"].sum()), int(e["n_entities"].sum())),
                     "CCA_%": pct(int(c["comp_correct"].sum()), len(c)),
                     "D": len(c)})
    return pd.DataFrame(rows)

cat_df = stage_table("category")
print(cat_df.to_string(index=False))
if "EQUATION_BASED_CALCS" in globals():
    eval_df["nature"] = eval_df["calculator"].apply(lambda c: "Equation" if c in EQUATION_BASED_CALCS else "Rule-based")
    nature_df = stage_table("nature")
    print(nature_df.to_string(index=False))
    nature_df.to_csv(os.path.join(RESULTS_DIR, "nature_wise_results.csv"), index=False)

# ---- save
os.makedirs(RESULTS_DIR, exist_ok=True)
pd.DataFrame(error_records).to_csv(os.path.join(RESULTS_DIR, "error_analysis.csv"), index=False)
with open(os.path.join(RESULTS_DIR, "error_summary.json"), "w") as f:
    json.dump({"error_counts": error_counts, "total_errors": total_errors}, f, indent=4)
cat_df.to_csv(os.path.join(RESULTS_DIR, "category_results.csv"), index=False)
eval_df.to_csv(os.path.join(RESULTS_DIR, "per_sample_evaluation.csv"), index=False)
summary = {"dataset": DATASET_NAME, "model": MODEL_ID, "N": N,
           "S": n_selected, "CSA_%": CSA,
           "PEA_matched": int(ext_part["n_matched"].sum()), "PEA_entities": int(ext_part["n_entities"].sum()), "PEA_%": PEA,
           "CCA_correct": int(comp_part["comp_correct"].sum()), "CCA_denominator": len(comp_part), "CCA_%": CCA}
if "technical_errors" in globals():
    summary["technical_errors"] = int(technical_errors)   # MCP only
with open(os.path.join(RESULTS_DIR, "overall_summary.json"), "w") as f:
    json.dump(summary, f, indent=4)
pd.DataFrame([summary]).to_csv(os.path.join(RESULTS_DIR, "All_combined_results.csv"), index=False)
print("Results saved in", RESULTS_DIR)